In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import itertools
import datetime
import wfdb
from scipy.signal import resample_poly
from functools import partial
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
)
from sklearn.dummy import DummyClassifier
from typing import Protocol, Any, Callable, Dict, List, Tuple
import plotly.express as px

class ECGSignalProcessor:
    def __init__(self, path="ptb-diagnostic-ecg-database-1.0.0"):
        self.PATH = path
        self.records_df = None
        self.all_signals = []
        self.all_fields = []
        self.Xy_trainvalid = None
        self.feature_columns = []
        self.label_column = "diagnosis"
        self.metrics = {
            "confusion_matrix": confusion_matrix,
            "accuracy": accuracy_score,
            "balanced_accuracy": balanced_accuracy_score,
            "matthews_corrcoef": matthews_corrcoef,
        }
        
    def load_data(self):
        PATH = self.PATH
        records_path = os.path.join(PATH, "RECORDS")
        self.records_df = pd.read_csv(records_path, header=None, names=["name"])
        self.records_df["patient"] = self.records_df["name"].apply(lambda x: x.split("/")[0])
        self.records_df["id"] = self.records_df["name"].apply(lambda x: x.split("/")[-1])
        self.records_df["ending"] = self.records_df["id"].apply(lambda x: x[-3:])
        self.records_df["id"] = self.records_df["id"].apply(lambda x: x[:-3])

        all_signals = []
        all_fields = []
        for record_name_ in tqdm(self.records_df["name"]):
            record_path = os.path.join(PATH, record_name_)
            signals, fields = wfdb.rdsamp(record_path)
            all_signals.append(signals)
            all_fields.append(fields)
        self.records_df["signals"] = all_signals
        self.records_df["fields"] = all_fields

    def preprocess_data(self):
        fields = self.records_df["fields"][0].keys()
        for field in fields:
            self.records_df[field] = self.records_df["fields"].apply(lambda x: x[field])
        self.records_df.drop(columns=["fields"], inplace=True)

        all_comments = itertools.chain(*self.records_df["comments"])
        all_comment_keys = sorted(set(x.split(":")[0] for x in all_comments))

        def no_start_whitespace(x):
            return x[1:] if x.startswith(" ") else x

        comment_dicts = [
            {comment.split(":")[0]: no_start_whitespace(comment.split(":")[1]) for comment in comments}
            for comments in self.records_df["comments"]
        ]

        for comment_key in all_comment_keys:
            self.records_df[comment_key] = [
                comment_dict.get(comment_key, None)
                for comment_dict in comment_dicts
            ]

        self.records_df.drop(columns=["comments"], inplace=True)

        for column in self.records_df.columns:
            if column == "signals":
                continue
            try:
                column_uniques = self.records_df[column].unique()
            except TypeError:
                column_uniques = self.records_df[column].drop_duplicates()
            if len(column_uniques) == 1:
                self.records_df.drop(columns=[column], inplace=True)

        self.records_df["age"] = pd.to_numeric(self.records_df["age"], errors="coerce")
        self.records_df.loc[self.records_df["sex"] == "", "sex"] = "n/a"
        self.records_df.rename(columns={"Smoker": "smoker"}, inplace=True)
        self.records_df.loc[self.records_df["smoker"] == "unknown", "smoker"] = "n/a"

        def parse_date(datestr):
            try:
                if "/" in datestr:
                    return datetime.datetime.strptime(datestr, "%d/%m/%Y")
                else:
                    return datetime.datetime.strptime(datestr, "%d-%b-%y")
            except ValueError:
                return pd.NaT
        
        self.records_df["ecg_date"] = self.records_df["ECG date"].apply(parse_date)
        self.records_df["admission_date"] = self.records_df["Admission date"].apply(parse_date)
        self.records_df["infarction_date"] = self.records_df["Infarction date"].apply(parse_date)

        self.records_df.rename(columns={"Reason for admission": "diagnosis"}, inplace=True)

        patient_df = self.records_df[["patient", "age", "sex", "smoker", "diagnosis"]].drop_duplicates()
        self.diagnosis_counts = patient_df["diagnosis"].value_counts()
        self.patient_fold_dict = {k: v for k, v in zip(patient_df["patient"], patient_df["fold"])}

    def evaluate_baseline_models(self):
        labels = sorted(label for label in self.diagnosis_counts[self.diagnosis_counts > 12].index if label != "n/a")

        # Rest of your code for evaluating baseline models with labels and selected patients
        # using class_weights, strategy_results, etc.
    def evaluate_baseline_models(self):
        baselines = ["most_frequent", "stratified", "uniform"]
        strategy_results = []
        for strategy in baselines:
            model = DummyClassifier(strategy=strategy)
            strategy_result = self.evaluate_folds(
                self.Xy_trainvalid,
                feature_columns=self.feature_columns,
                label_column=self.label_column,
                model=model,
                metrics=self.metrics,
            )
            strategy_result["strategy"] = strategy
            strategy_results.append(strategy_result)
        self.strategy_results = pd.concat(strategy_results)

    def augment_data(self, n_augmentations=3, window_length=8192):
        X = []
        for start_point in tqdm(np.linspace(0, 1, n_augmentations)):
            window_ds_signals = self.records_df["signals"].apply(
                lambda x: self.downsample(self.augment(x, start_point, window_length))
            )
            X.append(np.stack(window_ds_signals))
        X = np.concatenate(X)

        naive_df = pd.DataFrame(X[:, :, 0]).reset_index(drop=True)
        naive_df["diagnosis"] = np.tile(self.records_df["diagnosis"], n_augmentations)
        naive_df["patient"] = np.tile(self.records_df["patient"], n_augmentations)

        trainvalid_bools = naive_df["patient"].isin(self.patients_trainvalid["patient"])
        naive_df_trainvalid = naive_df[trainvalid_bools].copy()
        self.patient_fold_dict = {k: v for k, v in zip(self.patients_trainvalid["patient"], self.patients_trainvalid["fold"])}
        naive_df_trainvalid["fold"] = naive_df_trainvalid["patient"].apply(lambda x: self.patient_fold_dict[x])

        self.feature_columns = [i for i in range(1024)]

    def prepare_train_valid_data(self):
        # Your train/valid data preparation code here

    def train_and_evaluate(self):
        # Your training and evaluation code here

    def plot_sample_signal(self):
        # Your signal plotting code here

    def save_processed_data(self, output_file):
        # Your code to save processed data to a file here

    def load_processed_data(self, input_file):
        # Your code to load processed data from a file here
        pass

    def sex2float(self, x):
        # Your code for sex conversion here
        pass

    def smoker2float(self, x):
        # Your code for smoker conversion here
        pass

    def parse_date(self, datestr):
        # Your date parsing code here
        pass

    def augment(self, x, start, window_length):
        # Your data augmentation code here
        pass

    def process_signals(self):
        # Your signal processing code here
        pass

if __name__ == "__main__":
    processor = ECGSignalProcessor()
    processor.load_data()
    processor.preprocess_data()
    processor.evaluate_baseline_models()
    processor.augment_data()
    processor.prepare_train_valid_data()
    processor.train_and_evaluate()
    processor.plot_sample_signal()
    processor.save_processed_data("processed_data.csv")
    processor.load_processed_data("processed_data.csv")
